# Human vs. VARC Error Pattern Analysis
Compares error types across 400 ARC evaluation tasks.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

from analysis.load_data import (
    load_arc_ground_truth, load_varc_predictions,
    load_harc_summary, load_harc_incorrect_submissions
)
from analysis.error_analysis import (
    compute_varc_errors, compute_human_errors,
    task_level_summary, error_type_distribution
)

## 1. Load Data

In [ ]:
ground_truth     = load_arc_ground_truth()
varc_predictions = load_varc_predictions()
harc_df          = load_harc_summary()          # summary_data.csv, eval only
harc_agg         = load_harc_incorrect_submissions()  # incorrect_submissions.csv

print(f"ARC tasks:           {len(ground_truth)}")
print(f"VARC predictions:    {len(varc_predictions)}")
print(f"H-ARC attempts:      {len(harc_df)}")

In [ ]:
# Sanity check: inspect a few rows
harc_df[['task_id','hashed_id','attempt_number','solved','test_output_grid']].head(3)

## 2. Compute Errors

In [ ]:
varc_errors  = compute_varc_errors(ground_truth, varc_predictions)
human_errors = compute_human_errors(harc_df, ground_truth)

print("VARC  errors:",  varc_errors.shape,  "\n", varc_errors['error_type'].value_counts())
print("Human errors:", human_errors.shape, "\n", human_errors['error_type'].value_counts())

## 3. Error Type Distribution

In [ ]:
ERROR_ORDER = ['correct', 'close_miss', 'wrong_content', 'wrong_size']

varc_dist  = error_type_distribution(varc_errors).reindex(ERROR_ORDER, fill_value=0)
human_dist = error_type_distribution(human_errors).reindex(ERROR_ORDER, fill_value=0)

x = np.arange(len(ERROR_ORDER))
w = 0.35
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - w/2, human_dist.values, w, label='Human', color='steelblue', alpha=0.85)
ax.bar(x + w/2, varc_dist.values,  w, label='VARC',  color='tomato',   alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(ERROR_ORDER, fontsize=11)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_ylabel('Proportion of attempts')
ax.set_title('Error Type Distribution: Human vs. VARC')
ax.legend()
plt.tight_layout()
plt.savefig('../results/error_distribution.png', dpi=150)
plt.show()

pd.DataFrame({'Human': human_dist, 'VARC': varc_dist})

## 4. Task-Level Accuracy Comparison

In [ ]:
summary = task_level_summary(human_errors, varc_errors)

fig, ax = plt.subplots(figsize=(6, 6))
varc_jitter = summary['varc_correct'].astype(float) + np.random.uniform(-0.03, 0.03, len(summary))
ax.scatter(summary['human_accuracy'], varc_jitter, alpha=0.4, s=15, color='steelblue')
ax.axhline(0.5, color='gray', lw=0.8, ls='--')
ax.axvline(0.5, color='gray', lw=0.8, ls='--')
ax.set_xlabel('Human accuracy (fraction correct)')
ax.set_ylabel('VARC correct (jittered)')
ax.set_title('Per-Task Accuracy: Human vs. VARC')
plt.tight_layout()
plt.savefig('../results/task_accuracy_scatter.png', dpi=150)
plt.show()

q = {
    'Both correct':  ((summary.human_accuracy > 0.5) &  summary.varc_correct).sum(),
    'Human only':    ((summary.human_accuracy > 0.5) & ~summary.varc_correct).sum(),
    'VARC only':     ((summary.human_accuracy <= 0.5) &  summary.varc_correct).sum(),
    'Both wrong':    ((summary.human_accuracy <= 0.5) & ~summary.varc_correct).sum(),
}
pd.Series(q, name='tasks')

## 5. Cell Accuracy When Wrong

In [ ]:
varc_wrong  = varc_errors.loc[varc_errors.error_type  != 'correct', 'cell_accuracy'].dropna()
human_wrong = human_errors.loc[human_errors.error_type != 'correct', 'cell_accuracy'].dropna()

fig, ax = plt.subplots(figsize=(7, 4))
bins = np.linspace(0, 1, 21)
ax.hist(human_wrong, bins=bins, alpha=0.6, label=f'Human (n={len(human_wrong)})', color='steelblue', density=True)
ax.hist(varc_wrong,  bins=bins, alpha=0.6, label=f'VARC  (n={len(varc_wrong)})',  color='tomato',   density=True)
ax.set_xlabel('Cell accuracy on incorrect attempts')
ax.set_ylabel('Density')
ax.set_title('How Wrong Are Wrong Answers?')
ax.legend()
plt.tight_layout()
plt.savefig('../results/cell_accuracy_dist.png', dpi=150)
plt.show()

print('Mean cell accuracy (wrong only):')
print(f'  Human: {human_wrong.mean():.3f}   VARC: {varc_wrong.mean():.3f}')

## 6. Most Common Human Wrong Answers (from incorrect_submissions.csv)

In [ ]:
# Top 10 tasks with the most distinct wrong answers submitted by humans
diverse_errors = (
    harc_agg.groupby('task_id')
    .agg(unique_wrong_answers=('test_output_grid', 'count'),
         total_wrong_submissions=('count', 'sum'))
    .sort_values('total_wrong_submissions', ascending=False)
    .head(10)
)
diverse_errors

## 7. Save Results

In [ ]:
varc_errors.to_csv('../results/varc_errors.csv', index=False)
human_errors.to_csv('../results/human_errors.csv', index=False)
summary.to_csv('../results/task_summary.csv', index=False)
print('Saved to results/')